# S10. XML · JSON 다루기 — 공공데이터 API 응답 읽기

| 항목 | 내용 |
|---|---|
| 권장 시기 | 10주차 (19회 이후) |
| 시간 | 60분 |
| 변환한 R 코드 | `4. R/3. 코드 연습/1. 일자별 연습/Test/함수_XML.R` |

**XML**과 **JSON**은 웹·공공데이터 API가 데이터를 보내 주는 **글자 형식**입니다. 6단계(웹 데이터 가져오기)의 준비 운동입니다.
이 노트북은 인터넷 없이 연습하도록 **가상 응답**을 글자로 넣어 두었습니다. (구조는 공공데이터포털의 대기오염 API 응답과 비슷하게 만들었습니다)

## 1. XML — 태그로 감싼 데이터
```xml
<item>
  <stationName>가상동</stationName>
  <pm10Value>48</pm10Value>
</item>
```
- `<태그>값</태그>` 구조, 태그 안에 태그가 들어가는 **나무 모양**
- R 코드는 `xml2::read_xml`, `xml_children`, `xml_find_all`, `xml_text`를 썼고, 파이썬은 표준 라이브러리 **`xml.etree.ElementTree`** 를 씁니다.

In [ ]:
xml_text = '''<?xml version="1.0" encoding="UTF-8"?>
<response>
  <header><resultCode>00</resultCode><resultMsg>NORMAL</resultMsg></header>
  <body>
    <items>
      <item><stationName>가상동</stationName><dataTime>2025-04-08 10:00</dataTime><pm10Value>48</pm10Value><pm25Value>23</pm25Value><o3Value>0.031</o3Value></item>
      <item><stationName>나라동</stationName><dataTime>2025-04-08 10:00</dataTime><pm10Value>-</pm10Value><pm25Value>31</pm25Value><o3Value>0.028</o3Value></item>
      <item><stationName>다솜읍</stationName><dataTime>2025-04-08 10:00</dataTime><pm10Value>95</pm10Value><pm25Value>41</pm25Value><o3Value>0.045</o3Value></item>
    </items>
    <totalCount>3</totalCount>
  </body>
</response>'''
print(xml_text[:200])

In [ ]:
import xml.etree.ElementTree as ET

root = ET.fromstring(xml_text)                  # R의 read_xml
print("결과 코드:", root.find("header/resultCode").text)
print("전체 건수:", root.find("body/totalCount").text)

items = root.findall("body/items/item")         # R의 xml_find_all
print("item 수:", len(items))
for item in items:
    print(item.find("stationName").text, item.find("pm10Value").text)

### XML → 딕셔너리의 리스트 (표)
R 코드의 `tibble(key = xml_name(), value = xml_text()) %>% spread(key, value)`와 같은 일입니다.

In [ ]:
rows = []
for item in items:
    row = {child.tag: child.text for child in item}     # 자식 태그 이름 → 값
    rows.append(row)

for r in rows:
    print(r)

### 🧪 연습 1
`rows`의 값은 모두 **문자열**이고, 측정이 안 된 값은 `"-"` 입니다. (공공데이터에서 자주 보는 표시)
20회의 `to_number`를 써서 `pm10Value`, `pm25Value`, `o3Value`를 숫자(또는 `None`)로 바꾸고, **PM10 평균(결측 제외)** 을 구하세요.

In [ ]:
def to_number(text):
    try:
        return float(text.strip().replace(",", ""))
    except (ValueError, AttributeError):
        return None
# ✍️ 여기에 코드를 작성하세요

## 2. JSON — 파이썬 딕셔너리와 거의 같은 모양
요즘 API는 대부분 JSON을 씁니다. `json.loads`(글자 → 파이썬), `json.dumps`(파이썬 → 글자)

In [ ]:
import json

json_text = """
{"response": {"header": {"resultCode": "00"},
  "body": {"totalCount": 2,
    "items": [
      {"stationName": "가상동", "dataTime": "2025-04-08 11:00", "pm10Value": "52", "pm25Value": "25"},
      {"stationName": "다솜읍", "dataTime": "2025-04-08 11:00", "pm10Value": "101", "pm25Value": "44"}
    ]}}}
"""
data = json.loads(json_text)
items = data["response"]["body"]["items"]          # 딕셔너리처럼 키로 들어가기
for it in items:
    print(it["stationName"], it["pm10Value"])

### 🧪 연습 2
JSON의 `items`를 반복하며 **PM2.5 등급**(15회, 예시 구간: ≤15 좋음, ≤35 보통, ≤75 나쁨, 그 외 매우나쁨)을 붙인 새 딕셔너리 리스트를 만들고,
`output_air.json` 파일로 **한글이 보이게** 저장한 뒤 다시 읽어 출력하세요. (힌트: `json.dump(..., ensure_ascii=False, indent=2)`)

In [ ]:
# ✍️ 여기에 코드를 작성하세요

### 🧪 연습 3 — AI와 함께
AI에게 물어보세요:
> "공공데이터포털 API에서 XML과 JSON 중 어떤 형식으로 받는 게 파이썬 초보자에게 편해? 이유와, requests 라이브러리로 JSON을 받아 딕셔너리로 바꾸는 기본 코드 예시를 보여 줘. 인증키는 '내_인증키'로 표시해 줘."

받은 코드는 **실행하지 말고**(인증키가 필요), 아래 칸에 붙여 넣은 뒤 각 줄에 무엇을 하는지 주석을 다세요. 6단계 39회에서 실제로 실행합니다.

In [ ]:
# ✍️ 여기에 코드를 작성하세요

---
## 정리
| | XML | JSON |
|---|---|---|
| 모양 | `<태그>값</태그>` | `{"키": 값}` |
| 파이썬 도구 | `xml.etree.ElementTree` | `json` |
| 읽기 | `ET.fromstring(글자)` → `.find()`, `.findall()`, `.text` | `json.loads(글자)` → 딕셔너리 |
| 저장 | — | `json.dump(데이터, f, ensure_ascii=False, indent=2)` |
| 주의 | 값은 모두 **문자열**, `"-"` 같은 결측 표시 | 숫자도 문자열로 오는 경우 많음 |